# Site-Safe fire/smoke detector: training on Kaggle

Same recipe as `train_fire_smoke_colab.ipynb` (YOLO11s, 640 px, 100 epochs, AdamW, lr 0.001, seed 42, patience 20), built to finish unattended with **Save Version → Save & Run All**:

- Everything that must be kept is written to `/kaggle/working`, which Kaggle saves as the version's output.
- Training stops by itself after `MAX_TRAIN_HOURS` (checked after each saved epoch), leaving time for the held-out test and the export inside Kaggle's session limit. A time-stopped run is recorded as such, never hidden.
- The dataset is first copied to local disk in parallel: Kaggle unzips uploads into slow, read-only network storage, and Ultralytics repairs truncated JPEGs in place (read-only images would be dropped and the data would differ from earlier runs).
- The split is checked against the counts of the earlier local and Colab runs before any training starts.

**Inputs** (one private Kaggle dataset): `D-Fire.zip` (or its extracted folder) and `prepare_fire_dataset.py` from `ai-module/fire/`. **Settings:** Accelerator *GPU T4 x2* (one GPU is used), Internet *On*.

**Outputs:** `fire_smoke_v1_export/` (weights, checksum, metrics, run manifest) and `fire_smoke_v1_export.zip`.

## 1. Dependencies (needs Internet On)

In [ ]:
%pip install -q "ultralytics==8.4.41" "PyYAML==6.0.3"

## 2. Settings and GPU check

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

NOTEBOOK_START = time.time()
INPUT = Path(os.environ.get("FIRE_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("FIRE_WORK", "/kaggle/working"))
SCRATCH = Path(os.environ.get("FIRE_SCRATCH", "/tmp"))
DRY_RUN = os.environ.get("FIRE_DRY_RUN") == "1"  # tiny local smoke test of this notebook, not a result

RUN_NAME = "fire_smoke_v1"
SOURCE_CLASSES = "smoke,fire"  # D-Fire's class ids: 0=smoke, 1=fire
# Kaggle ends a GPU session at 12 hours. Training stops after this many hours,
# leaving time for setup (~15 min), the held-out test and the export.
MAX_TRAIN_HOURS = float(os.environ.get("FIRE_MAX_TRAIN_HOURS", "10.0"))
# Counts from the local (22 Oct) and Colab runs of prepare_fire_dataset.py on D-Fire.
EXPECTED_COUNTS = {"train": 15469, "val": 1738, "test": 4294}

import torch
if not DRY_RUN:
    assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU in the notebook settings"
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| python", sys.version.split()[0], "| dry run" if DRY_RUN else "")

## 3. Find the inputs

In [ ]:
# os.walk with followlinks: Kaggle attaches datasets as symlinks, which
# Path.rglob does not enter on Python < 3.13.
def walk(base):
    for root, dirs, files in os.walk(base, followlinks=True):
        dirs[:] = sorted(d for d in dirs if d != "__MACOSX")
        yield Path(root), dirs, sorted(files)

def find_one(name_test, base=INPUT):
    for root, _, files in walk(base):
        for f in files:
            if name_test(f):
                return root / f
    return None

def dataset_root(base):
    # Folder holding train/images and test/images (D-Fire's layout).
    for root, dirs, _ in walk(base):
        if (root / "train" / "images").is_dir() and (root / "test" / "images").is_dir():
            return root
    return None

print("Attached inputs:")
for root, dirs, files in walk(INPUT):
    depth = len(root.relative_to(INPUT).parts)
    if depth <= 2:
        print("  " * depth + f"{root.name}/  ({len(files)} files: {', '.join(files[:5])}{' ...' if len(files) > 5 else ''})")

PREPARE = find_one(lambda f: f == "prepare_fire_dataset.py")
assert PREPARE, f"prepare_fire_dataset.py not found under {INPUT}: see the listing above and add the file to the dataset"

RAW = dataset_root(INPUT)
if RAW is not None:
    # Kaggle unzipped the upload. Reading ~43k small files one by one from its network
    # storage takes very long, so copy them to local disk in parallel first. Local copies
    # are also writable, which Ultralytics needs to repair truncated JPEGs.
    from concurrent.futures import ThreadPoolExecutor
    staged = SCRATCH / "fire_raw"
    files = [root / f for root, _, names in walk(RAW) for f in names]
    print(f"copying {len(files)} files from {RAW} to {staged}", flush=True)
    def copy_one(src):
        dst = staged / src.relative_to(RAW)
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            shutil.copyfile(src, dst)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=32) as pool:
        for i, _ in enumerate(pool.map(copy_one, files), 1):
            if i % 2000 == 0 or i == len(files):
                print(f"  {i}/{len(files)} files, {time.time() - t0:.0f} s", flush=True)
    RAW = staged
else:
    archive = find_one(lambda f: f.lower().endswith(".zip"))
    assert archive, f"No D-Fire folder or zip found under {INPUT}: see the listing above"
    target = SCRATCH / "fire_raw"
    if not target.exists():
        print("extracting", archive)
        with zipfile.ZipFile(archive) as z:
            z.extractall(target)
    RAW = dataset_root(target)
    assert RAW, f"{archive} has no train/images and test/images folders"
print("prepare script:", PREPARE)
print("dataset root:  ", RAW)

## 4. Validate, remap and split
Same script and seed as the earlier runs; stops if the split differs.

In [ ]:
DATASET = SCRATCH / "fire_dataset_v1"
if DATASET.exists():
    shutil.rmtree(DATASET)
# RAW is on local, writable disk (section 3), so the prepared split can symlink to it.
subprocess.run([sys.executable, str(PREPARE), "--source", str(RAW), "--source-classes", SOURCE_CLASSES,
                "--out", str(DATASET)], check=True)
manifest = json.loads((DATASET / "manifest.json").read_text())
counts = {split: c["images"] for split, c in manifest["counts"].items()}
print(json.dumps(manifest["counts"], indent=2))
print("skipped images:", len(manifest["skipped_images"]))
if not DRY_RUN:
    assert counts == EXPECTED_COUNTS, f"Split differs from earlier runs: {counts} != {EXPECTED_COUNTS}"
DATA_YAML = DATASET / "data.yaml"

## 5. Train
About 5 minutes per epoch on a T4, so 100 epochs take about 8.5 hours.

In [ ]:
import random
import numpy as np
from ultralytics import YOLO

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
PROJECT = WORK / "runs" / "fire"
TRAIN_ARGS = dict(data=str(DATA_YAML), epochs=100, imgsz=640, batch=16, patience=20, optimizer="AdamW",
                  lr0=0.001, seed=SEED, deterministic=True, device=0, workers=4,
                  project=str(PROJECT), name=RUN_NAME, exist_ok=False, plots=True)
if DRY_RUN:
    TRAIN_ARGS.update(epochs=2, fraction=1.0, batch=8, workers=0, device="cpu", deterministic=False)

guard = {"stopped_by_time_guard": False}
def stop_before_session_limit(trainer):
    """Runs after each epoch is validated and saved; ending here keeps last.pt and best.pt intact."""
    hours = (time.time() - TRAIN_START) / 3600
    if hours > MAX_TRAIN_HOURS and not trainer.stop:
        trainer.stop = True
        guard.update(stopped_by_time_guard=True, stopped_after_epoch=trainer.epoch + 1, hours=round(hours, 2))
        print(f"Time guard: stopping after epoch {trainer.epoch + 1} ({hours:.2f} h of training)")

model = YOLO("yolo11s.pt")
model.add_callback("on_fit_epoch_end", stop_before_session_limit)
TRAIN_START = time.time()
model.train(**TRAIN_ARGS)
TRAIN_SECONDS = round(time.time() - TRAIN_START)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
EPOCHS_DONE = int(model.trainer.epoch) + 1
assert BEST.is_file(), "training produced no best.pt"
print(f"{EPOCHS_DONE} epochs in {TRAIN_SECONDS / 3600:.2f} h; guard: {guard}")

## 6. Held-out test (run once)

In [ ]:
# Held-out test split: evaluated once, never used for training or model selection.
metrics = YOLO(str(BEST)).val(data=str(DATA_YAML), split="test", imgsz=640, batch=TRAIN_ARGS["batch"],
                              device=TRAIN_ARGS["device"], project=str(WORK / "runs" / "fire"),
                              name=f"{RUN_NAME}_held_out", exist_ok=True, plots=True)
names = metrics.names
box = metrics.box
results = {
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "split": "test",
    "overall": {"precision": float(box.mp), "recall": float(box.mr), "map50": float(box.map50), "map50_95": float(box.map)},
    "per_class": {names[int(c)]: {"precision": float(p), "recall": float(r), "map50": float(m50), "map50_95": float(m)}
                  for c, p, r, m50, m in zip(box.ap_class_index, box.p, box.r, box.ap50, box.ap)},
    "test_counts": manifest["counts"].get("test", {}),
}
print(json.dumps(results["overall"], indent=2))

## 7. Export
Image-level mAP is not alert quality; evaluate alerts on recorded clips, including hard negatives, before claiming results.

In [ ]:
import hashlib, platform
import ultralytics

EXPORT = WORK / f"{RUN_NAME}_export"
if EXPORT.exists():
    shutil.rmtree(EXPORT)
EXPORT.mkdir(parents=True)
shutil.copy2(BEST, EXPORT / "fire_smoke.pt")
for name in ("args.yaml", "results.csv", "results.png", "confusion_matrix.png"):
    if (RUN_DIR / name).exists():
        shutil.copy2(RUN_DIR / name, EXPORT / name)
shutil.copy2(DATASET / "manifest.json", EXPORT / "dataset_manifest.json")
(EXPORT / "held_out_test_metrics.json").write_text(json.dumps(results, indent=2))
sha = hashlib.sha256((EXPORT / "fire_smoke.pt").read_bytes()).hexdigest()
run = {
    "status": "training_and_held_out_test_completed",
    "platform": "kaggle" if not DRY_RUN else "local dry run (not a result)",
    "model_source": "yolo11s.pt",
    "train_args": {k: v for k, v in TRAIN_ARGS.items()},
    "epochs_completed": EPOCHS_DONE,
    "training_seconds": TRAIN_SECONDS,
    "time_guard": {"max_train_hours": MAX_TRAIN_HOURS, **guard},
    "weights_sha256": sha,
    "environment": {"python": platform.python_version(), "ultralytics": ultralytics.__version__, "torch": torch.__version__,
                    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"},
    "dataset": {"counts": manifest["counts"], "skipped_images": len(manifest["skipped_images"])},
    "notebook_seconds": round(time.time() - NOTEBOOK_START),
    "finished_at_utc": datetime.now(timezone.utc).isoformat(),
}
(EXPORT / "run_manifest.json").write_text(json.dumps(run, indent=2))
shutil.make_archive(str(EXPORT), "zip", EXPORT)
print("sha256", sha)
print("download", EXPORT.with_suffix(".zip"))
print("DONE")